In [1]:
import os
import time
import uuid
from pathlib import Path

from pypdf import PdfReader
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

In [2]:
from dotenv import load_dotenv
import os
load_dotenv()

PDF_PATH = r"2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf"

INDEX_NAME = "indigo-book-rag"
NAMESPACE = "economy-report"

EMBED_MODEL = "text-embedding-3-small"
GEN_MODEL = "gpt-4o-mini"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 4

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)

In [3]:
def load_pdf_text(pdf_path: str) -> list[dict]:
    reader = PdfReader(pdf_path)
    docs = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            docs.append({
                "page": i + 1,
                "text": text.strip()
            })
    return docs

In [4]:
def split_text(text: str, chunk_size: int = 800, overlap: int = 150) -> list[str]:
    text = " ".join(text.split())
    chunks = []

    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        if end >= len(text):
            break
        start = end - overlap

    return chunks

In [5]:
def build_chunks(docs: list[dict]) -> list[dict]:
    all_chunks = []

    for doc in docs:
        page = doc["page"]
        text = doc["text"]

        chunks = split_text(text, CHUNK_SIZE, CHUNK_OVERLAP)
        for idx, chunk in enumerate(chunks):
            all_chunks.append({
                "id": f"page-{page}-chunk-{idx}-{uuid.uuid4().hex[:8]}",
                "page": page,
                "chunk_id": idx,
                "text": chunk
            })

    return all_chunks

In [6]:
def embed_texts(texts: list[str], batch_size: int = 64) -> list[list[float]]:
    vectors = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(
            model=EMBED_MODEL,
            input=batch
        )
        batch_vectors = [item.embedding for item in resp.data]
        vectors.extend(batch_vectors)

    return vectors

In [7]:
def get_embedding_dimension() -> int:
    resp = client.embeddings.create(
        model=EMBED_MODEL,
        input=["dimension check"]
    )
    return len(resp.data[0].embedding)

In [8]:
def ensure_index(index_name: str):
    existing_indexes = [idx["name"] for idx in pc.list_indexes()]

    if index_name in existing_indexes:
        print(f"이미 인덱스가 존재합니다: {index_name}")
        return

    dim = get_embedding_dimension()

    pc.create_index(
        name=index_name,
        dimension=dim,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

    print(f"인덱스 생성 요청 완료: {index_name}")

    while True:
        desc = pc.describe_index(index_name)
        status = desc.status["ready"]
        if status:
            break
        time.sleep(2)

    print("인덱스 준비 완료")

In [9]:
def ingest_pdf_to_pinecone(pdf_path: str, index_name: str, namespace: str):
    docs = load_pdf_text(pdf_path)
    chunks = build_chunks(docs)
    texts = [c["text"] for c in chunks]
    embeddings = embed_texts(texts)

    index = get_index(index_name)

    batch = []
    for chunk, embedding in zip(chunks, embeddings):
        batch.append({
            "id": chunk["id"],
            "values": embedding,
            "metadata": {
                "page": chunk["page"],
                "chunk_id": chunk["chunk_id"],
                "text": chunk["text"]
            }
        })

    batch_size = 100
    for i in range(0, len(batch), batch_size):
        index.upsert(
            vectors=batch[i:i + batch_size],
            namespace=namespace
        )
        print(f"업서트 진행: {min(i + batch_size, len(batch))}/{len(batch)}")

    print("업서트 완료")

In [10]:
def get_index(index_name: str):
    desc = pc.describe_index(index_name)
    host = desc.host
    return pc.Index(host=host)

In [11]:
def ingest_pdf_to_pinecone(pdf_path: str, index_name: str, namespace: str):
    docs = load_pdf_text(pdf_path)
    chunks = build_chunks(docs)
    texts = [c["text"] for c in chunks]
    embeddings = embed_texts(texts)

    index = get_index(index_name)

    batch = []
    for chunk, embedding in zip(chunks, embeddings):
        batch.append({
            "id": chunk["id"],
            "values": embedding,
            "metadata": {
                "page": chunk["page"],
                "chunk_id": chunk["chunk_id"],
                "text": chunk["text"]
            }
        })

    batch_size = 100
    for i in range(0, len(batch), batch_size):
        index.upsert(
            vectors=batch[i:i + batch_size],
            namespace=namespace
        )
        print(f"업서트 진행: {min(i + batch_size, len(batch))}/{len(batch)}")

    print("업서트 완료")

In [12]:
ensure_index(INDEX_NAME)
ingest_pdf_to_pinecone(PDF_PATH, INDEX_NAME, NAMESPACE)

이미 인덱스가 존재합니다: indigo-book-rag


KeyboardInterrupt: 

In [ ]:
def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    index = get_index(INDEX_NAME)

    q_emb = client.embeddings.create(
        model=EMBED_MODEL,
        input=[query]
    )
    query_vector = q_emb.data[0].embedding

    result = index.query(
        namespace=NAMESPACE,
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    matches = []
    for match in result["matches"]:
        page_value = match["metadata"]["page"]

        matches.append({
            "id": match["id"],
            "score": match["score"],
            "page": int(float(page_value)),   # 여기 수정
            "chunk_id": int(float(match["metadata"]["chunk_id"])),
            "text": match["metadata"]["text"]
        })

    return matches

In [ ]:
results = retrieve("2026년 한국 GDP 성장률 전망은?", top_k=4)

for r in results:
    print(f"page={r['page']}, score={r['score']:.4f}")
    print(r["text"][:300])
    print("-" * 80)

In [ ]:
def answer_question(query: str, top_k: int = TOP_K) -> str:
    retrieved = retrieve(query, top_k=top_k)

    context_blocks = []
    pages = []

    for i, item in enumerate(retrieved, 1):
        pages.append(item["page"])
        context_blocks.append(
            f"[근거 {i}] (page {item['page']})\n{item['text']}"
        )

    context = "\n\n".join(context_blocks)
    page_note = ", ".join(map(str, sorted(set(pages))))

    prompt = f"""
너는 업로드된 PDF 문서만 근거로 답하는 한국어 경제 보고서 QA 도우미다.

규칙:
- 반드시 아래 검색 문맥에 근거해서만 답하라.
- 문맥에 없으면 "문서에서 확인되지 않습니다."라고 말하라.
- 추측하지 말라.
- 답변 마지막에 참고 페이지를 적어라.
- 핵심만 한국어로 정리하라.

[사용자 질문]
{query}

[검색 문맥]
{context}
"""

    resp = client.responses.create(
        model=GEN_MODEL,
        input=prompt
    )

    return f"{resp.output_text}\n\n참고 페이지: {page_note}"

In [ ]:
answer = answer_question("2026년 한국 GDP 성장률 전망은 몇 퍼센트야?")
print(answer)

In [ ]:
while True:
    q = input("\n질문 입력 (종료: exit) > ").strip()
    if q.lower() == "exit":
        break

    try:
        ans = answer_question(q, top_k=4)
        print("\n[답변]")
        print(ans)

        print("\n[검색 근거]")
        for r in retrieve(q, top_k=4):
            print(f"- page {r['page']} | score={r['score']:.4f}")
            print(r["text"][:200], "...")
            print()

    except Exception as e:
        print(f"오류 발생: {e}")